# Network Intrusion Detection Using Random Forest and Deep Neural Networks

**Roll Numbers:** 2520030378, 2520030350, 2520030356
**Department:** Computer Science Engineering, KLH University

This notebook builds a machine learning system that classifies network traffic as
**normal** or **anomaly** (an intrusion), using a **Random Forest** classifier and a
**Deep Neural Network (DNN)**. We compare both models on Accuracy, Precision, Recall,
F1-score, and ROC-AUC, as described in the project abstract.


## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Load the Dataset

We use `Train_data.csv` (the NSL-KDD dataset), which contains 25,192 network
connection records with 41 traffic features described in the abstract: protocol
type, service, flag, byte counts, connection counts, error rates, host-based
statistics, and more, plus the `class` label (`normal` or `anomaly`).


In [ ]:
filename = "Train_data.csv"

# If the file is already sitting in this Colab session's disk (e.g. you uploaded it
# earlier and only cleared the output, or you added it via the Files sidebar), this
# skips straight to reading it -- no upload prompt. It only asks you to choose the
# file the first time in a fresh runtime.
if not os.path.exists(filename):
    from google.colab import files
    uploaded = files.upload()   # click "Choose Files" and select Train_data.csv
    filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print("Dataset shape:", df.shape)
df.head()


In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()

## 3. Exploratory Data Analysis (EDA)

Before modeling, we look at the class distribution and how key features relate to
whether a connection is normal or an intrusion.


In [ ]:
plt.figure()
sns.countplot(x='class', data=df, palette=['#2E86AB', '#E63946'])
plt.title('Class Distribution (normal vs anomaly)')
plt.xlabel('Class')
plt.ylabel('Number of Connections')
plt.show()

print(df['class'].value_counts(normalize=True) * 100)


In [ ]:
plt.figure()
sns.countplot(x='protocol_type', hue='class', data=df, palette=['#2E86AB', '#E63946'])
plt.title('Class Count by Protocol Type')
plt.show()


In [ ]:
plt.figure()
sns.histplot(data=df, x='src_bytes', hue='class', bins=30, log_scale=(True, False),
             palette=['#2E86AB', '#E63946'])
plt.title('Source Bytes Distribution by Class')
plt.show()


In [ ]:
plt.figure(figsize=(12,9))
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()
sns.heatmap(corr, cmap='coolwarm', xticklabels=False, yticklabels=False)
plt.title('Correlation Heatmap (numeric features)')
plt.show()


## 4. Data Preprocessing

Steps:
1. Encode the categorical features `protocol_type`, `service`, `flag` using Label Encoding.
2. Encode the target `class` (`normal` = 0, `anomaly` = 1).
3. Split into features (X) and target (y).
4. Train/test split (80/20), stratified so both sets keep the same class ratio.
5. Scale features with `StandardScaler`.


In [ ]:
data = df.copy()

categorical_cols = ['protocol_type', 'service', 'flag']
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

target_le = LabelEncoder()
data['class'] = target_le.fit_transform(data['class'])  # normal=0, anomaly=1

X = data.drop(columns=['class'])
y = data['class']

print("Final feature columns:", list(X.columns))
X.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", X_train.shape, " Test size:", X_test.shape)


## 5. Model 1 — Random Forest

Random Forest builds many decision trees on random subsets of data/features and
averages their votes. It handles a large number of features well and gives us
interpretable feature importances.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, random_state=SEED, n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

rf_pred = rf.predict(X_test_scaled)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]


## 6. Model 2 — Deep Neural Network

The DNN is a fully connected feed-forward network. It learns complex, non-linear
patterns across the traffic features that a tree-based model might miss. Dropout
layers reduce overfitting, and `EarlyStopping` halts training once validation loss
stops improving.


In [ ]:
n_features = X_train_scaled.shape[1]

dnn = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = dnn.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
dnn_proba = dnn.predict(X_test_scaled).ravel()
dnn_pred = (dnn_proba >= 0.5).astype(int)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('DNN Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('DNN Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Model Evaluation

We compare both models using Accuracy, Precision, Recall, F1-score, and ROC-AUC —
exactly the metrics named in the abstract.


In [ ]:
def get_metrics(name, y_true, y_pred, y_proba):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

results = pd.DataFrame([
    get_metrics('Random Forest', y_test, rf_pred, rf_proba),
    get_metrics('Deep Neural Network', y_test, dnn_pred, dnn_proba)
]).set_index('Model')

results.round(4)


In [ ]:
results.round(3).plot(kind='bar', figsize=(10,6), colormap='viridis')
plt.title('Random Forest vs DNN — Metric Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, pred) in zip(axes, [('Random Forest', rf_pred), ('Deep Neural Network', dnn_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['normal', 'anomaly'], yticklabels=['normal', 'anomaly'])
    ax.set_title(f'{name} — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


### ROC Curves

In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_dnn, tpr_dnn, _ = roc_curve(y_test, dnn_proba)

plt.figure()
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_proba):.3f})")
plt.plot(fpr_dnn, tpr_dnn, label=f"Deep Neural Network (AUC = {roc_auc_score(y_test, dnn_proba):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()


### Detailed Classification Reports

In [ ]:
print("Random Forest:\n", classification_report(y_test, rf_pred, target_names=['normal','anomaly']))
print("Deep Neural Network:\n", classification_report(y_test, dnn_pred, target_names=['normal','anomaly']))


## 8. Feature Importance

Which traffic features matter most for detecting intrusions? Random Forest gives us
this directly. (The DNN doesn't expose native feature importances the same way, since
it learns distributed, non-linear combinations of inputs rather than per-feature splits.)


In [ ]:
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,6))
rf_importance.head(15).plot(kind='barh', color='#2E86AB')
plt.gca().invert_yaxis()
plt.title('Random Forest — Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 9. Conclusion

- Both **Random Forest** and a **Deep Neural Network** were trained on 25,192 NSL-KDD
  network connection records using the features described in the abstract (protocol
  type, service, flag, byte counts, connection/error rates, host-based statistics, and
  more).
- Model performance was evaluated using Accuracy, Precision, Recall, F1-score, and
  ROC-AUC (see the comparison table/chart in Section 7).
- Random Forest offers a fast, interpretable baseline with clear feature importances,
  while the DNN can capture more complex, non-linear traffic patterns given enough
  tuning.
- The results above (metrics, confusion matrices, ROC curves, classification reports)
  can be pasted directly into the project report's Results section.


## 10. Which specific connections are most likely to be intrusions?

The metrics above tell us how *good* the models are overall. But in practice, a
security team wants an actual list: "here are the connections most likely to be
attacks, go investigate them." This is done using `predict_proba()`, which gives a
**risk score between 0 and 1** for every connection (0 = definitely normal, 1 =
definitely an intrusion) instead of just a plain yes/no.


In [ ]:
# Build a results table for the test connections (the unseen 20%)
risk_table = X_test.copy()
risk_table['Actual_Class'] = y_test.values
risk_table['RF_Risk_Score'] = rf_proba          # Random Forest's risk score (0 to 1)
risk_table['DNN_Risk_Score'] = dnn_proba        # DNN's risk score (0 to 1)

# Show ALL connections the Random Forest model predicts are intrusions
all_predicted_anomalies = risk_table[risk_table['RF_Risk_Score'] >= 0.5].sort_values(
    'RF_Risk_Score', ascending=False
)

all_predicted_anomalies[['Actual_Class', 'RF_Risk_Score', 'DNN_Risk_Score']].head(20)


**How to read this table:** each row is a real connection from our test set.
`RF_Risk_Score` close to 1.0 means the model is very confident this connection is an
intrusion. `Actual_Class` shows whether it actually was an anomaly (1) or normal (0) —
so you can see for yourself how often the model's top-risk list matches reality.

In a real deployment, a security team would take this exact table, sort by risk
score, and investigate the top rows before an attack causes damage.


In [ ]:
# How many of the FULL predicted-anomaly list actually were intrusions?
full_hit_rate = all_predicted_anomalies['Actual_Class'].sum() / len(all_predicted_anomalies) * 100
print(f"Of all {len(all_predicted_anomalies)} connections the model flagged as likely intrusions, "
      f"{all_predicted_anomalies['Actual_Class'].sum()} out of {len(all_predicted_anomalies)} "
      f"({full_hit_rate:.0f}%) actually were anomalies.")


### Full test-set view (not just flagged connections)

The table above only shows connections the model flagged as likely intrusions. For a
complete picture -- e.g. if a reviewer wants to see everything, including connections
the model was confident were normal -- here is the full test set with both models'
risk scores attached to every single row.


In [ ]:
# Full test set (all 5,038 rows), with both models' risk scores on every row
full_test_view = X_test.copy()
full_test_view['Actual_Class'] = y_test.map({0: 'normal', 1: 'anomaly'}).values
full_test_view['RF_Predicted'] = pd.Series(rf_pred, index=y_test.index).map({0: 'normal', 1: 'anomaly'})
full_test_view['RF_Risk_Score'] = rf_proba
full_test_view['DNN_Predicted'] = pd.Series(dnn_pred, index=y_test.index).map({0: 'normal', 1: 'anomaly'})
full_test_view['DNN_Risk_Score'] = dnn_proba

print("Full test set size:", full_test_view.shape[0])
full_test_view[['Actual_Class', 'RF_Predicted', 'RF_Risk_Score',
                'DNN_Predicted', 'DNN_Risk_Score']].sort_values('RF_Risk_Score', ascending=False)
